## Structured Error Analysis

The goal of structured error analysis is not only to measure performance numerically, but also to understand the types of mistakes made by each model. 

In particular, this analysis focuses on:

- which classes are most commonly confused
- whether the models fail on the same examples
- what linguistic or contextual properties make certain tweets difficult to classify

This will help provide us with a deeper insight into model behavior beyond overall accuracy and macro-averaged metrics.

### Load saved Artifacts

In [2]:
import joblib

# Load train-test split
x_train = joblib.load(r".\Artifacts\x_train.pkl")
x_test = joblib.load(r".\Artifacts\x_test.pkl")
y_train = joblib.load(r".\Artifacts\y_train.pkl")
y_test = joblib.load(r".\Artifacts\y_test.pkl")
class_names = joblib.load(r".\Artifacts\class_names.pkl")

print("Data split loaded")

# Load Naive Bayes
nb_model = joblib.load(r".\Artifacts\naive_bayes_model.pkl")
nb_vectorizer = joblib.load(r".\Artifacts\naive_bayes_vectorizer.pkl")

# Load Logistic Regression
lr_model = joblib.load(r".\Artifacts\logistic_regression_model.pkl")
lr_vectorizer = joblib.load(r".\Artifacts\logistic_regression_vectorizer.pkl")

# Load SVM
svm_model = joblib.load(r".\Artifacts\svm_model.pkl")
svm_vectorizer = joblib.load(r".\Artifacts\svm_vectorizer.pkl")

print("NB, LR, SVM model loaded")
print("NB, LR, SVM vectorizer loaded")

Data split loaded
NB, LR, SVM model loaded
NB, LR, SVM vectorizer loaded


### Recreate Predictions on the Test Set

In [7]:
models = {
    "NB": (nb_model, nb_vectorizer),
    "LR": (lr_model, lr_vectorizer),
    "SVM": (svm_model, svm_vectorizer)
}

predictions = {}

for name, (model, vectorizer) in models.items():
    x_test_vecs = vectorizer.transform(x_test)
    predictions[name] = model.predict(x_test_vecs)
    print(f"{name} predictions generated")

NB predictions generated
LR predictions generated
SVM predictions generated


## Master Error Analysis Table

This table consolidates the ground truth labels, model predictions, and correctness indicators across all models at the instance level.

It serves as the foundation for structured error analysis by enabling:

- Identification of systematic misclassification patterns (e.g., specific label confusions)
- Direct comparison of model behaviors on the same input instances
- Isolation of cases where models agree or disagree in their errors
- Extraction of representative error samples for qualitative analysis

This unified view allows us to move beyond aggregate metrics (e.g., accuracy, precision) and instead analyze **how** and **why** models fail.

In [8]:
import pandas as pd

error_df = pd.DataFrame({
    "tweet": x_test,
    "true_label": [class_names[i] for i in y_test]
})

for model_name, y_pred in predictions.items():
    preds = []
    for i in y_pred:
        preds.append(class_names[i])
        
    error_df[f"{model_name}_pred"] = preds
    error_df[f"{model_name}_correct"] = error_df["true_label"] == error_df[f"{model_name}_pred"]

error_df.head()

,tweet,true_label,NB_pred,NB_correct,LR_pred,LR_correct,SVM_pred,SVM_correct
0,today tomorrow try move mountains cross fingers,other_cyberbullying,not_cyberbullying,False,not_cyberbullying,False,other_cyberbullying,True
1,typa girl bullied lesbians high school,age,age,True,age,True,age,True
2,cool woman gave directions polling place whose...,age,age,True,age,True,age,True
3,need teach version quran terrorists killing pe...,religion,religion,True,religion,True,religion,True
4,rome destroy coliseum roman forum every buildi...,religion,religion,True,religion,True,religion,True


### Error Partitioning Analysis

To better understand model behavior beyond aggregate performance metrics, we partition the dataset based on how many models correctly classify each sample.

For each tweet, we compute the number of models that predict the correct label. Using this, we divide the dataset into three categories:

- **All Correct**: Tweets correctly classified by all models  
- **All Wrong**: Tweets misclassified by all models  
- **Mixed**: Tweets where some models are correct and others are incorrect  

This partitioning allows us to approximate the difficulty of the dataset:

- Tweets in the *All Correct* category are likely easier and contain clear patterns that all models can learn.  
- Tweets in the *All Wrong* category represent difficult or ambiguous cases that current models consistently fail to capture.  
- The *Mixed* category is particularly important, as it highlights disagreement between models and reveals differences in their learning behavior.

By structuring the data in this way, we move from evaluating overall accuracy to analyzing where and why models succeed or fail, which forms the basis for deeper error analysis in subsequent sections.

In [14]:
model_names = list(predictions.keys())

# Count how many models got it correct per row
error_df["num_correct"] = error_df[[f"{m}_correct" for m in model_names]].sum(axis=1)

# Count how many got it wrong
error_df["num_wrong"] = len(model_names) - error_df["num_correct"]

# All models correct
all_correct = error_df[error_df["num_correct"] == len(model_names)]

# All models wrong
all_wrong = error_df[error_df["num_correct"] == 0]

# Mixed (some correct, some wrong)
mixed = error_df[
    (error_df["num_correct"] > 0) &
    (error_df["num_correct"] < len(model_names))
]

model_strengths = {}

for m in model_names:
    model_strengths[m] = error_df[
        (error_df[f"{m}_correct"] == True) &
        (error_df["num_correct"] < len(model_names))  # others failed
    ]

model_weaknesses = {}

for m in model_names:
    model_weaknesses[m] = error_df[
        (error_df[f"{m}_correct"] == False) &
        (error_df["num_correct"] > 0)  # others succeeded
    ]

summary = pd.DataFrame({
    "Total Samples": [len(error_df)],
    "All Correct": [len(all_correct)],
    "All Wrong": [len(all_wrong)],
    "Mixed": [len(mixed)]
})

summary

,Total Samples,All Correct,All Wrong,Mixed
0,11923,8174,1622,2127


**Observations from Error Partitioning**

1) A majority of tweets (8174) are correctly classified by all models. 
   - This indicates that a large portion of the dataset contains clear and distinguishable patterns that are consistently captured across different modeling approaches.

2) However, 1622 tweets are misclassified by all models. 
   - This suggests the presence of inherently difficult cases which could be due to ambiguity, subtle language, or weak class specific signals. 
   - The fact that all models fail on these cases indicates that the issue lies more with the data complexity than with a specific model.

3) Notably, 2127 tweets fall into the mixed category, where models disagree in their predictions. 
   - These cases highlight the differences in model behavior, suggesting that each model captures different aspects of the data. 
   - For example, lexical cues vs contextual patterns. 
   - This also implies that no single model is universally optimal, and that combining models or using more expressive architectures may improve performance.

This partitioning provides a structured foundation for subsequent analysis by distinguishing between universally easy, universally hard, and model-dependent cases.

### Misclassification Pattern Analysis

To systematically identify where models fail, we analyze misclassification patterns by grouping incorrect predictions based on their true and predicted labels.

For each model, we compute the frequency of confusion pairs (true label → predicted label), allowing us to:

- Identify the most common types of misclassification  
- Detect systematic confusion between specific categories  
- Compare how different models struggle with similar label pairs  

By focusing on the most frequent confusion patterns, we can prioritize key areas for deeper qualitative analysis in the next section.

In [40]:
confusion_patterns = {}

for m in model_names:
    wrong_cases = error_df[error_df[f"{m}_correct"] == False]
    
    confusion_counts = (
        wrong_cases
        .groupby(["true_label", f"{m}_pred"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    
    confusion_patterns[m] = confusion_counts

top_confusions = {}

for m in model_names:
    top_confusions[m] = confusion_patterns[m].head(3)
    print(f"\n{'='*9} {m} Top 3 Misclassification {'='*9}")
    display(top_confusions[m])


========= NB Top 3 Misclassification =========


,true_label,NB_pred,count
20,other_cyberbullying,age,447
15,not_cyberbullying,age,442
18,not_cyberbullying,other_cyberbullying,385



========= LR Top 3 Misclassification =========


,true_label,LR_pred,count
18,not_cyberbullying,other_cyberbullying,713
23,other_cyberbullying,not_cyberbullying,566
12,gender,not_cyberbullying,163



========= SVM Top 3 Misclassification =========


,true_label,SVM_pred,count
17,not_cyberbullying,other_cyberbullying,865
22,other_cyberbullying,not_cyberbullying,473
12,gender,other_cyberbullying,196


*Observations from Misclassification Pattern Analysis*

1) `[Observation]` A dominant pattern across all models is the confusion between not_cyberbullying and other_cyberbullying.
   - `[Evidence]` Logistic Regression (713 & 566) and SVM (865 & 473) showing particularly high misclassification in both directions. 
   - `[Explanation]` This indicates that the boundary between non-abusive content and subtle or implicit cyberbullying is not clearly defined
   - Therefore, making these classes difficult to separate even for stronger models.

2) `[Observation]` Naive Bayes shows a strong bias towards predicting the age category. 
   - `[Evidence]` Misclassifying both not_cyberbullying (442) and other_cyberbullying (447) as age. 
   - `[Explanation]` This suggests that NB is heavily influenced by surface-level lexical features.
   - Likely overfitting to keywords associated with age-related language rather than capturing deeper contextual meaning.

3) `[Observation]` Gender-based cyberbullying is sometimes misclassified as not_cyberbullying or other_cyberbullying.
   - `[Evidence]` 163 cases as not_cyberbullying in LR or 196 cases as other_cyberbullying in SVM. 
   - `[Explanation]` This indicates that this class lacks strong or consistent distinguishing features. 
   - This may be due to variability in how gender-related abuse is expressed, or overlap with more general forms of offensive language.

More broadly, categories such as other_cyberbullying exhibit higher misclassification rates across all models. This suggests that these labels are semantically broad and overlap with multiple categories, making them inherently harder to model and more prone to misclassification. These patterns highlight that model errors are not random, but are concentrated in specific class boundaries and structurally ambiguous categories.

### Qualitative Error Analysis

While previous sections quantified error patterns and model disagreements, they do not fully explain the underlying reasons behind these misclassifications. To address this, we conduct a qualitative error analysis by examining representative tweet examples from the most frequent confusion pairs.

This analysis focuses on identifying recurring linguistic and contextual characteristics that lead to model errors. In particular, we investigate whether misclassifications arise due to factors such as implicit or indirect expressions of cyberbullying, ambiguous or weak class signals, short or context-limited text, and over-reliance on surface-level lexical features.

Through this approach, we aim to provide deeper insight into the limitations of the models and the challenges inherent in detecting cyberbullying in natural language.

In [43]:
def get_misclassification_examples(df, model_name, true_label, pred_label, n=5):
    subset = df[
        (df["true_label"] == true_label) &
        (df[f"{model_name}_pred"] == pred_label)
        ][["tweet", "true_label", f"{model_name}_pred"]]
    
    return subset.head(n)

In [42]:
for m in model_names:
    print(f"\n\n{'='*35} {m} {'='*35}")
    
    top3 = confusion_patterns[m].head(3)
    
    for _, row in top3.iterrows():
        true_label = row["true_label"]
        pred_label = row[f"{m}_pred"]
        count = row["count"]
        
        print(f"\nTrue: {true_label} | Predicted: {pred_label} | Count: {count}")
        display(get_misclassification_examples(error_df, m, true_label, pred_label, n=5))



=================================== NB ===================================

True: other_cyberbullying | Predicted: age | Count: 447


,tweet,true_label,NB_pred
7,pepperidge farm remembers,other_cyberbullying,age
13,lot people cell phones one years used idea dis...,other_cyberbullying,age
30,lol this person telling everybody bullying her...,other_cyberbullying,age
101,got one those seem hold tight enough,other_cyberbullying,age
106,people always trynah bully,other_cyberbullying,age



True: not_cyberbullying | Predicted: age | Count: 442


,tweet,true_label,NB_pred
6,city college,not_cyberbullying,age
15,fucking love her hot shit,not_cyberbullying,age
23,posthebdo lol events like daily occurrence aro...,not_cyberbullying,age
80,throw back fridays school schoolmetrofm,not_cyberbullying,age
117,every minutes child bullied adult intervention...,not_cyberbullying,age



True: not_cyberbullying | Predicted: other_cyberbullying | Count: 385


...

,tweet,true_label,SVM_pred
10,life better sex know sex know someone dying se...,not_cyberbullying,other_cyberbullying
15,fucking love her hot shit,not_cyberbullying,other_cyberbullying
36,hear smells wordsalad really justincoherent,not_cyberbullying,other_cyberbullying
37,color sky world,not_cyberbullying,other_cyberbullying
63,long winding story,not_cyberbullying,other_cyberbullying



True: other_cyberbullying | Predicted: not_cyberbullying | Count: 473


,tweet,true_label,SVM_pred
34,oops woke nap,other_cyberbullying,not_cyberbullying
47,still stage given finding good task list mgt a...,other_cyberbullying,not_cyberbullying
83,def weds,other_cyberbullying,not_cyberbullying
84,smell blocks morning,other_cyberbullying,not_cyberbullying
92,yup especially since start hour drive home mor...,other_cyberbullying,not_cyberbullying



True: gender | Predicted: other_cyberbullying | Count: 196


,tweet,true_label,SVM_pred
14,yours mine enforced men jail visitation these ...,gender,other_cyberbullying
89,pro anti blameonenotall promoting nothing divi...,gender,other_cyberbullying
102,please try get manð© really hate kill bitche...,gender,other_cyberbullying
168,hear lot fans complaining end dalekdoctor conf...,gender,other_cyberbullying
169,sick useless ass people culture stfu sick usel...,gender,other_cyberbullying


**Inference from Qualitative Error Analysis**

1) `[Inference]` Across all models, there is a strong tendency to rely on surface-level lexical cues such as profanity or emotionally charged words. 
   - `[Evidence]` For instance, tweets like "fucking love her hot shit" and "hear smells wordsalad really justincoherent" are misclassified as other_cyberbullying` 
   - `[Explanation]` This is despite lacking clear abusive intent. This indicates that models often associate strong language with cyberbullying, even when context does not support it.

2) `[Inference]` Naive Bayes demonstrates a clear over-reliance on isolated keywords, particularly for the `age` category. 
   - `[Evidence]` Examples such as "city college" and "throw back fridays school" are incorrectly classified as `age`.
   - `[Explanation]` This suggests that the model is triggered by school or age related terms without considering the broader semantic context.

3) `[Inference]` Gender based cyberbullying is difficult for models to detect consistently due to weak or ambiguous class signals. 
   - `[Evidence]` Tweets such as "poor ben" and "want home please rouge" lack explicit gender-related indicators, leading models to classify them as not_cyberbullying.
   - In other cases, tweets like "please try get man ... really hate kill bitches" contain strong offensive language but are still misclassified. 
   - `[Explanation]` This suggests that the models struggle to distinguish between general abuse and abuse that is specifically targeted at gender. 
   - This also indicates that identifying the target of the abuse is even more challenging than detecting the presence of offensive language.

4) `[Inference]` The misclassification between not_cyberbullying and other_cyberbullying is consistently reflected in the examples. 
   - `[Evidence]` Many tweets are short, ambiguous, or lack explicit intent. For example, "long winding story", "summer its...". 
   - `[Explanation]` This makes it difficult for models to determine whether the content is abusive. 
   - This also reinforces earlier findings that the boundary between these classes is inherently unclear.

More broadly, the misclassified examples highlight recurring challenges such as short or context poor text, implicit or indirect expressions of abuse, and semantically overlapping class definitions. These factors limit the ability of the models, especially those relying on bag-of-words or shallow representations to accurately capture nuanced meaning. These findings therefore confirm that the model errors are driven not just by noise, but also by systematic limitations in capturing context, intent, and nuanced linguistic patterns.

### Cross Model Behaviour Analysis

In [46]:
for m in model_names:
    correct = error_df[error_df[f"{m}_correct"] == True]["true_label"].value_counts()
    print(f"\n{m} correct predictions per class:\n", correct)


NB correct predictions per class:
 true_label
age                    1924
religion               1916
ethnicity              1814
gender                 1604
other_cyberbullying     712
not_cyberbullying       661
Name: count, dtype: int64

LR correct predictions per class:
 true_label
ethnicity              1946
age                    1899
religion               1870
gender                 1611
other_cyberbullying    1273
not_cyberbullying      1120
Name: count, dtype: int64

SVM correct predictions per class:
 true_label
ethnicity              1952
age                    1909
religion               1858
gender                 1581
other_cyberbullying    1414
not_cyberbullying       987
Name: count, dtype: int64


**Observations from Cross Model Behaviour Analysis**

- Classes such as `age`, `religion`, `ethnicity`, and `gender` exhibit consistently high correct predictions across all models, with only minor variation. This suggests that these categories contain strong and explicit lexical cues, making them easier to classify regardless of model complexity.

- In contrast, `other_cyberbullying` and `not_cyberbullying` show significantly lower performance, indicating that these classes are inherently more ambiguous and lack clear distinguishing features.

- Naive Bayes performs substantially worse than Logistic Regression and SVM on these difficult classes, highlighting its limitation in capturing contextual nuances due to its reliance on independent word frequency assumptions.

- SVM achieves the highest performance on `other_cyberbullying` but performs worse than Logistic Regression on `not_cyberbullying`, suggesting that SVM may be more inclined to classify borderline cases as cyberbullying, whereas Logistic Regression maintains a more balanced decision boundary.

- The largest discrepancies across models occur in distinguishing between `not_cyberbullying` and `other_cyberbullying`, indicating that the primary challenge lies in boundary ambiguity rather than identifying specific categories.

- These findings are consistent with the misclassification pattern analysis, where these two classes are frequently misclassified into each other. Qualitative analysis further shows that many errors involve implicit aggression or general offensive language without a clear target.

- Finally, the performance gain from Naive Bayes to Logistic Regression is significantly larger than the gain from Logistic Regression to SVM, suggesting that discriminative linear models already capture most of the useful patterns in the dataset.

## Final Insights and Conclusion

### Dataset Difficulty

A large proportion of tweets are classified correctly by all the models. This shows that many tweets contain clear and explicit signals. However, the presence of mixed and universally misclassified samples shows that a significant portion of the dataset is inherently ambiguous and difficult to model.

---

### Key Source of Error

The primary challenge across all models lies in distinguishing between `not_cyberbullying` and `other_cyberbullying`. These two classes exhibit the highest misclassification rates, suggesting that their boundaries are not well-defined.

Qualitative analysis shows that this ambiguity arises from:
- emotionally charged but non-abusive language (false positives)
- subtle or implicit cyberbullying without explicit indicators (false negatives)

---

### Model Behaviour

Naive Bayes tends to rely heavily on individual keywords, leading to systematic misclassification when certain lexical cues (e.g., age-related terms) are present.

Logistic Regression and SVM perform better overall, but still struggle with ambiguous and context-dependent cases. The relatively small performance gap between them suggests that most useful patterns are already captured by linear models.

---

### Class-Level Observations

Classes such as `age`, `religion`, `ethnicity`, and `gender` are consistently classified well, indicating the presence of strong and distinctive features.

In contrast, `not_cyberbullying` and `other_cyberbullying` remain the most challenging due to overlapping definitions and weaker signals.

---

### Overall Insight

These findings highlight that the main limitation lies not only in model design, but also in the inherent complexity of the task. Cyberbullying detection often depends on subtle context, intent, and nuance, which are difficult to capture from isolated text.

---

### Future Direction

Improving performance may require:
- more context-aware models (e.g., transformer-based architectures)
- clearer class definitions
- incorporation of additional contextual information

Ultimately, better performance will require both stronger models and a deeper understanding of linguistic complexity.